# Gravity exponent $\gamma$ sweep — full pipeline & static-impact comparison (2017)

Reconstructs the intra-US MRIOT with the **gravity model** for five values of the friction exponent
$\gamma \in \{0.1,\ 0.5,\ 1,\ 1.5,\ 3\}$, runs the **entire pipeline** for each
($\text{build (v3.1)} \to \text{column RAS} \to \text{VA convention} \to \text{sectoral aggregation} \to \text{harmonization} \to \text{nesting into OECD ICIO}$),
then compares:

1. the **propagation losses** of a regional shock under the **Leontief**, **Ghosh** and **IIM** models, and
2. the **structure** of the reconstructed table (localization of interstate trade) across $\gamma$.

The bilateral friction is $d_{ij}^{-\gamma}$: a **small $\gamma$** lets goods travel far (diffuse, near-uniform
interstate trade), a **large $\gamma$** concentrates trade between nearby states (localized trade).
All five runs use the exact v3.1_RAS construction code (`gamma_sweep.py`, validated to reproduce the
production `v3.1_RAS` build bit-for-bit at $\gamma=1$); only $\gamma$ changes
($\gamma_{\text{trade}}=\gamma_{\text{margin}}=\gamma$).

## Configuration & setup

In [ ]:
import sys
import pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))   # local modules: gamma_sweep, harmonize, nest_v31

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gamma_sweep as gs

YEAR         = 2017
GAMMAS       = [0.1, 0.5, 1.0, 1.5, 3.0]
SHOCK_REGION = 'NY'      # state hit by the shock
THETA        = 0.30     # fraction of activity / demand / capacity lost

# Load the WiNDC GDX once and build the SAGDP2 state-share pivot (used by nesting).
gs.setup()
shares_pivot = gs.nest_v31.build_shares_pivot()
print('ready — gammas:', GAMMAS)

## 1. Run the full pipeline for each $\gamma$

For every $\gamma$ this builds the state-level table, RAS-balances it, applies the VA convention,
aggregates the sectors, harmonizes to the OECD US aggregates and nests the result into the OECD ICIO
world table. Everything is kept **in memory** (`results[g]`): nesting needs the harmonized table on
disk, so it is written to a throwaway temp dir and removed immediately — nothing is persisted to the
project tree (the five runs together would be ~1.2 GB). Pass `save=True` to additionally write the
standard `grav_fric_gamma<g>_RAS[...]` / `nested_mriot_gamma<g>` artifacts (needs free quota).
Expect ~30 s per $\gamma$.

In [ ]:
results = {}
for g in GAMMAS:
    results[g] = gs.run_pipeline_for_gamma(YEAR, g, shares_pivot)   # save=True to persist artifacts
print('\nall pipelines done. nested shape:', results[GAMMAS[0]]['nested'].shape)

## 2. Static impact models — Leontief, Ghosh, IIM

On each nested table we shock the state `SHOCK_REGION` by `THETA` and propagate it:

| model | shock | mechanism | direction |
|---|---|---|---|
| **Leontief** | final demand $-\theta$ | $\Delta x = L\,\Delta f$ | upstream (suppliers) |
| **IIM** | inoperability $=\theta$ | $q = \hat{x}^{-1} L\,(\hat{x} c)$, loss $= q\odot x$ | upstream (inoperability) |
| **Ghosh** | primary inputs $-\theta$ | $\Delta x' = \Delta v'\,G$ | downstream (clients) |

We report the loss split into the shocked state (direct), the rest of the US and the rest of the world
(spillovers).

In [ ]:
impacts = {}
for g in GAMMAS:
    blocks = gs.load_nested_blocks(results[g]['nested'])
    impacts[g] = gs.compute_impacts(blocks, SHOCK_REGION, THETA)

# Sanity: the table must reproduce itself (L f = x and v' G = x).
repro = pd.DataFrame({g: impacts[g]['repro'] for g in GAMMAS}).T
repro.index.name = 'gamma'
print('reproduction error (max |.|, M$):')
repro

In [ ]:
# Loss summary table: one row per (gamma, model).
rows = []
for g in GAMMAS:
    summ = impacts[g]['summary']
    for model in ('Leontief', 'IIM', 'Ghosh'):
        s = summ[model]
        rows.append({
            'gamma': g, 'model': model,
            f'{SHOCK_REGION}_direct': s[f'{SHOCK_REGION}_direct'],
            'other_US': s['other_US_spillover'],
            'world': s['world_spillover'],
            'TOTAL': s['TOTAL'],
        })
loss_df = pd.DataFrame(rows)
# spillover outside the shocked state, in % of the direct effect (scale-free)
loss_df['spillover_pct_of_direct'] = (
    (loss_df['other_US'] + loss_df['world']) / loss_df[f'{SHOCK_REGION}_direct'] * 100)
pd.set_option('display.float_format', lambda v: f'{v:,.0f}')
print(f'Loss decomposition (M$), shock = {SHOCK_REGION} -{THETA:.0%}')
loss_df

In [ ]:
# Pivot views: total loss and out-of-state spillover (% of direct) vs gamma.
tot = loss_df.pivot(index='gamma', columns='model', values='TOTAL')
spill = loss_df.pivot(index='gamma', columns='model', values='spillover_pct_of_direct')
print('TOTAL loss (M$) by gamma x model'); display(tot.round(0))
print('\nOut-of-state spillover (% of direct effect) by gamma x model'); display(spill.round(2))

In [ ]:
models = ['Leontief', 'IIM', 'Ghosh']
colors = {'Leontief': '#1f77b4', 'IIM': '#d62728', 'Ghosh': '#2ca02c'}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) total loss vs gamma
for m in models:
    axes[0].plot(GAMMAS, np.abs(tot[m]), 'o-', color=colors[m], label=m)
axes[0].set_xlabel(r'$\gamma$'); axes[0].set_ylabel('|total loss| (M$)')
axes[0].set_title('Total propagated loss vs $\\gamma$'); axes[0].legend(); axes[0].grid(alpha=.3)

# (b) out-of-state spillover (% of direct) vs gamma
for m in models:
    axes[1].plot(GAMMAS, spill[m], 'o-', color=colors[m], label=m)
axes[1].set_xlabel(r'$\gamma$'); axes[1].set_ylabel('spillover (% of direct)')
axes[1].set_title('Out-of-state spillover vs $\\gamma$'); axes[1].legend(); axes[1].grid(alpha=.3)

# (c) US vs world spillover split (Leontief), stacked
lt = loss_df[loss_df.model == 'Leontief'].set_index('gamma')
axes[2].bar([str(g) for g in GAMMAS], np.abs(lt['other_US']), label='other US', color='#1f77b4')
axes[2].bar([str(g) for g in GAMMAS], np.abs(lt['world']), bottom=np.abs(lt['other_US']),
            label='rest of world', color='#ff7f0e')
axes[2].set_xlabel(r'$\gamma$'); axes[2].set_ylabel('|spillover| (M$)')
axes[2].set_title('Leontief spillover split vs $\\gamma$'); axes[2].legend()
plt.tight_layout(); plt.show()

## 3. Structural comparison of the reconstructed table across $\gamma$

How $\gamma$ reshapes the intra-US **state $\times$ state** intermediate-flow picture (before nesting):

- **intra_share** — fraction of intermediate flows that stay within the producing state;
- **mean_trade_distance_km** — flow-weighted mean shipping distance of interstate trade;
- **interstate_gini** — spatial concentration of the off-diagonal (state$\to$state) flow matrix.

Note that the intra- vs inter-state *split* is essentially **pinned by the WiNDC marginals + the RAS
balancing** (the row/column totals of each sector's bilateral block are fixed by `xn0`/`nd0`/`nm0`
regardless of $\gamma$, and the local `dd0`/`dm0` flows are $\gamma$-independent), so `intra_share`
barely moves. What $\gamma$ controls is the **spatial allocation within the interstate block**: a higher
$\gamma$ pushes flows onto nearby state pairs — mean trade distance $\downarrow$ and concentration
(Gini) $\uparrow$.

In [ ]:
struct = {g: gs.structural_metrics(results[g]['table']) for g in GAMMAS}
str_df = pd.DataFrame([
    {'gamma': g, **{k: v for k, v in struct[g].items() if k != 'Z_rs'}}
    for g in GAMMAS]).set_index('gamma')
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('Structural metrics of the intra-US Z block vs gamma')
str_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, ttl in [
    (axes[0], 'inter_share', 'Interstate share of intermediate flows'),
    (axes[1], 'mean_trade_distance_km', 'Mean interstate trade distance (km)'),
    (axes[2], 'interstate_gini', 'Concentration of interstate flows (Gini)')]:
    ax.plot(GAMMAS, str_df[col], 'o-', color='#6a3d9a')
    ax.set_xlabel(r'$\gamma$'); ax.set_title(ttl); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# State x state interstate intermediate-flow matrix (log), for the two extreme gammas.
states = list(results[GAMMAS[0]]['table']['regions'])
g_lo, g_hi = GAMMAS[0], GAMMAS[-1]
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, g in zip(axes, (g_lo, g_hi)):
    M = struct[g]['Z_rs'].copy()
    np.fill_diagonal(M, 0.0)                      # interstate only
    im = ax.imshow(np.log10(M + 1e-3), cmap='viridis', aspect='auto')
    ax.set_title(f'$\\gamma={g}$ — interstate intermediate flows (log10 M$)')
    ax.set_xticks(range(len(states))); ax.set_xticklabels(states, rotation=90, fontsize=5)
    ax.set_yticks(range(len(states))); ax.set_yticklabels(states, fontsize=5)
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()
print(f'Low gamma ({g_lo}) = diffuse long-range trade;  high gamma ({g_hi}) = localized near-diagonal trade.')

In [ ]:
# How far the whole table moves vs the gamma=1 reference (relative L1 distance).
def rel_l1(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.abs(a - b).sum() / (np.abs(b).sum() or 1.0)

ref = results[1.0]
drift = pd.DataFrame([{
    'gamma': g,
    'Z_state_level': rel_l1(results[g]['table']['Z'], ref['table']['Z']),
    'Z_nested': rel_l1(results[g]['nested'].loc[:, :].values, ref['nested'].values),
} for g in GAMMAS]).set_index('gamma')
print('Relative L1 distance of the table to the gamma=1 build')
drift

## 4. Per-sector $\gamma$ scenarios — random & economically-structured frictions

Sections 1–3 vary a **single scalar** $\gamma$ shared by all 71 build sectors. Here we instead give
**each sector its own friction exponent** $\gamma_g$ (still $\gamma_{\text{trade},g}=\gamma_{\text{margin},g}$)
and run the *same* full pipeline for several per-sector $\gamma$ vectors:

- **`economic`** — an economically-motivated assignment: **small** $\gamma$ for *service / light* sectors
  (weightless output, cheap to "ship" far) and **large** $\gamma$ for *heavy / material* sectors
  (manufacturing, mining, agriculture, primary metals, construction, utilities — costly to move), with
  transport in between.
- **`structured_rand`** — the same service/transport/heavy ordering, but each $\gamma_g$ is drawn at
  random *within* its band, so the qualitative structure is kept while the exact values vary.
- **`random_0/1/2`** — three fully **random** draws, $\gamma_g \sim \mathcal{U}(0, 3)$ independently per
  sector, as unstructured controls.

Sectors are classed service / transport / heavy from the OECD *proposed sector* each WiNDC code maps to.
All scenarios are compared against the **scalar $\gamma=1$** baseline from sections 1–3.

In [ ]:
# --- Classify each of the 71 build sectors: service (light) / transport / heavy-material ---
w2p = gs.build_windc_to_proposed()
HEAVY_KEYS = ('Manufacture', 'Mining', 'Oil and gas', 'Agriculture',
              'Construction', 'Electricity', 'primary metals')

def sector_class(code):
    prop = w2p.get(code, '')
    if any(k in prop for k in HEAVY_KEYS):                    return 'heavy'
    if 'transport' in prop.lower() or 'Warehousing' in prop:  return 'transport'
    return 'service'

SECTORS = list(gs.sectors)
classes = {s: sector_class(s) for s in SECTORS}
print('sector counts by class:',
      {c: sum(v == c for v in classes.values()) for c in ('service', 'transport', 'heavy')})

def to_vec(fn):
    """Build a length-len(gs.sectors) gamma vector aligned with gs.sectors order."""
    return np.array([fn(s) for s in SECTORS], dtype=float)

# (1) economic, deterministic: small gamma for services, large for heavy, medium transport.
ECON  = {'service': 0.3, 'transport': 1.2, 'heavy': 2.5}
# (2) structured-random: same ordering, gamma_g drawn within each class's band.
BANDS = {'service': (0.1, 0.8), 'transport': (0.8, 1.6), 'heavy': (1.8, 3.0)}

def draw_structured(seed):
    rng = np.random.default_rng(seed)
    return to_vec(lambda s: rng.uniform(*BANDS[classes[s]]))

def draw_random(seed):                       # fully random control: U(0, 3) per sector
    return np.random.default_rng(seed).uniform(0.0, 3.0, len(SECTORS))

scen_gammas = {
    'economic':        to_vec(lambda s: ECON[classes[s]]),
    'structured_rand': draw_structured(seed=1),
    'random_0':        draw_random(seed=10),
    'random_1':        draw_random(seed=11),
    'random_2':        draw_random(seed=12),
}

# Tabulate the gamma each scenario assigns to each sector.
gamma_table = pd.DataFrame({'class': [classes[s] for s in SECTORS]},
                           index=pd.Index(SECTORS, name='sector'))
for name, vec in scen_gammas.items():
    gamma_table[name] = vec
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
print('mean gamma per scenario x class:')
display(gamma_table.groupby('class').mean())
gamma_table.sort_values('class')

In [ ]:
# Run the full build -> RAS -> VA conv -> aggregate -> harmonize -> nest pipeline per
# scenario (~30 s each). Pass the scenario name as `label` so each per-sector vector
# (which has no single gamma value) gets a readable on-disk / in-memory tag.
scen_results = {}
for name, vec in scen_gammas.items():
    scen_results[name] = gs.run_pipeline_for_gamma(YEAR, vec, shares_pivot, label=name)
print('\nall per-sector scenarios done. nested shape:',
      scen_results['economic']['nested'].shape)

### Impact & structure comparison

Same static-impact shock as section 2 (`SHOCK_REGION` $-\theta$) and the same structural metrics as
section 3, now reported **per scenario** and against the scalar $\gamma=1$ baseline.

In [ ]:
# Static-impact losses per scenario, with the scalar gamma=1 build as reference.
scen_impacts = {}
for name in scen_gammas:
    blocks = gs.load_nested_blocks(scen_results[name]['nested'])
    scen_impacts[name] = gs.compute_impacts(blocks, SHOCK_REGION, THETA)

def _loss_rows(name, imp):
    out = []
    for model in ('Leontief', 'IIM', 'Ghosh'):
        s = imp['summary'][model]
        out.append({'scenario': name, 'model': model,
                    f'{SHOCK_REGION}_direct': s[f'{SHOCK_REGION}_direct'],
                    'other_US': s['other_US_spillover'],
                    'world': s['world_spillover'], 'TOTAL': s['TOTAL']})
    return out

rows = _loss_rows('scalar_gamma1', impacts[1.0])          # baseline from section 2
for name in scen_gammas:
    rows += _loss_rows(name, scen_impacts[name])
scen_loss = pd.DataFrame(rows)
scen_loss['spillover_pct_of_direct'] = (
    (scen_loss['other_US'] + scen_loss['world']) / scen_loss[f'{SHOCK_REGION}_direct'] * 100)
pd.set_option('display.float_format', lambda v: f'{v:,.0f}')
print(f'Loss decomposition (M$), shock = {SHOCK_REGION} -{THETA:.0%}')
scen_loss

In [ ]:
ORDER  = ['scalar_gamma1', 'economic', 'structured_rand', 'random_0', 'random_1', 'random_2']
MODELS = ['Leontief', 'IIM', 'Ghosh']
PARTS  = [('NY_direct', f'{SHOCK_REGION} direct', '#1f77b4'),
          ('other_US',  'other US',               '#ff7f0e'),
          ('world',     'rest of world',          '#2ca02c')]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, model in zip(axes, MODELS):
    d = scen_loss[scen_loss.model == model].set_index('scenario').reindex(ORDER)
    bottom = np.zeros(len(ORDER))
    for col, lab, color in PARTS:
        vals = np.abs(d[col].values)
        ax.bar(ORDER, vals, bottom=bottom, label=lab, color=color)
        bottom += vals
    ax.set_title(f'{model} — loss decomposition by $\\gamma$ vector')
    ax.set_ylabel('|loss| (M$)')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=.3, axis='y')
axes[0].legend()
plt.tight_layout(); plt.show()


In [ ]:
SCEN   = ['economic', 'structured_rand', 'random_0', 'random_1', 'random_2']  # γ1 = 0 ref, omis
MODELS = ['Leontief', 'IIM', 'Ghosh']
PARTS  = [('NY_direct', f'{SHOCK_REGION} direct', '#1f77b4'),
          ('other_US',  'other US',               '#ff7f0e'),
          ('world',     'rest of world',          '#2ca02c'),
          ('TOTAL',     'total',                  '#444444')]

x = np.arange(len(SCEN)); w = 0.8 / len(PARTS)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, model in zip(axes, MODELS):
    d   = scen_loss[scen_loss.model == model].set_index('scenario')
    ref = d.loc['scalar_gamma1']                      # γ=1 baseline
    for k, (col, lab, color) in enumerate(PARTS):
        dev = (d.loc[SCEN, col] - ref[col]) / ref[col] * 100   # % deviation from γ=1
        ax.bar(x + k * w, dev.values, w, label=lab, color=color)
    ax.axhline(0, color='k', lw=.8)
    ax.set_xticks(x + 0.4 - w / 2); ax.set_xticklabels(SCEN, rotation=30)
    ax.set_title(f'{model} — deviation from $\\gamma=1$')
    ax.set_ylabel('Δ loss vs $\\gamma=1$ (%)')
    ax.grid(alpha=.3, axis='y')
axes[0].legend()
plt.tight_layout(); plt.show()


In [ ]:
# Structural metrics of the intra-US Z block per scenario (+ scalar gamma=1 reference).
scen_struct = {name: gs.structural_metrics(scen_results[name]['table']) for name in scen_gammas}
scen_str_df = pd.DataFrame([
    {'scenario': name, **{k: v for k, v in scen_struct[name].items() if k != 'Z_rs'}}
    for name in scen_gammas]).set_index('scenario')
scen_str_df.loc['scalar_gamma1'] = {k: v for k, v in struct[1.0].items() if k != 'Z_rs'}
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('Structural metrics of the intra-US Z block by scenario')
scen_str_df

In [ ]:
ORDER = ['scalar_gamma1', 'economic', 'structured_rand', 'random_0', 'random_1', 'random_2']
lt = scen_loss[scen_loss.model == 'Leontief'].set_index('scenario').reindex(ORDER)
sd = scen_str_df.reindex(ORDER)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(ORDER, lt['spillover_pct_of_direct'], color='#1f77b4')
axes[0].set_ylabel('spillover (% of direct)')
axes[0].set_title('Leontief out-of-state spillover by scenario')
axes[1].bar(ORDER, sd['mean_trade_distance_km'], color='#6a3d9a')
axes[1].set_title('Mean interstate trade distance (km)')
axes[2].bar(ORDER, sd['interstate_gini'], color='#2ca02c')
axes[2].set_title('Concentration of interstate flows (Gini)')
for ax in axes:
    ax.tick_params(axis='x', rotation=30); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.show()

## 5. Takeaways

Results for the 2017 build, NY shocked by $-30\%$ (numbers from the sweep):

- **Structure.** As $\gamma$ grows $0.1\to3$, the friction $d_{ij}^{-\gamma}$ steepens and interstate
  trade **localizes**: flow-weighted mean trade distance falls $\approx 1750\to 805$ km and the
  interstate-flow Gini rises $\approx 0.64\to 0.86$. The intra- vs inter-state *split* stays $\approx 0.45/0.55$
  (pinned by the WiNDC marginals + RAS), so $\gamma$ acts purely on *where* the interstate flows go, not
  on *how much* is interstate — visible as flow mass migrating onto the near-diagonal in the heatmaps.
- **Impacts.** Higher $\gamma$ makes each state source more locally, so a regional shock stays closer to
  home: the **direct** NY loss grows (e.g. Leontief $-523\text{k}\to-538$k M\$) while the **other-US
  spillover shrinks monotonically** (Leontief $-184\text{k}\to-173$k; IIM $-350\text{k}\to-325$k;
  Ghosh $-255\text{k}\to-236$k M\$). The rest-of-world spillover and the grand total are nearly
  $\gamma$-invariant — $\gamma$ mainly **redistributes** the loss between the shocked state and the rest
  of the US, rather than changing its magnitude.
- The $\gamma=1$ run reproduces the production `v3.1_RAS` state-level build bit-for-bit, so these are
  clean *ceteris-paribus* comparisons isolating the single gravity parameter.
- **Per-sector $\gamma$ (section 4).** Relaxing the single-$\gamma$ assumption — small $\gamma$ for
  services, large for heavy/material sectors (`economic`), or random per sector (`random_*`) — places
  the structure and spillover *between* the uniform-$\gamma$ extremes: the aggregate response is a
  sector-mix-weighted blend of the per-sector frictions. The `economic` scenario localizes the
  goods-producing sectors that dominate interstate trade while leaving weightless services diffuse, so
  its structural metrics sit close to a moderately-high uniform $\gamma$, whereas the unstructured
  `random_*` draws average out toward the $\gamma\approx1.5$ middle. (Re-run the section to read off the
  exact figures for this build.)